# 1) Import libraries and Define User Parameters

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../modules')

import matplotlib.pyplot as plt
import matplotlib.animation as anim
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.signal import butter, filtfilt
plt.rcParams['figure.dpi'] = 150
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['animation.embed_limit'] = 2**128


from Class_sem2dpack import *
from Stage_module import *

#--------------------------------------------------
#                 USER PARAMETERS
#--------------------------------------------------

# Computation of the elastic parameters
cp = 700
cs = 300
rho = 2000
ratio = cp/cs
nu = (ratio**2 - 2)/(2*(ratio**2 - 1))
E = rho*(3*cp**2 - 4*cs**2)/(ratio**2 - 1)
G = E/(2*(1 + nu))
K = E/(3*(1 - 2*nu))
print(f"nu = {nu:.3e}")
print(f"E = {E:.3e} Pa")
print(f"G = {G:.3e} Pa")
print(f"K = {K:.3e} Pa")
# Computation of the friction parameters
fric_coefficient = 0.3
Phi = np.rad2deg(np.arctan(fric_coefficient))
print(f"Phi = {Phi:.2f}°")

# Computation of the Goodman model for stiffnesses
t_j = 6 # Joint thickness
dx = 6.66 # Elment size
Kn = E/t_j # Normal stiffness
Ks = G/t_j # Shear stiffness
Kn_max = 10*(K+4/3*G)/dx
print(f"Kn = {Kn:.3e} N/m")
print(f"Ks = {Ks:.3e} N/m")
print(f"Kn_max = {Kn_max:.3e} N/m")


#Kn = 2.5e7 # Normal stiffness
#Ks = 1e9 # Shear stiffness
#Phi = 16.7 # Friction angle
#--------------------------------------------------



# 2) Read Data

### Import SEM2DPACK Data

In [ ]:
# Directory name with output files
direct1 = "C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/OUTPUT/Stage_ISS_v10_verticalinci_stations"
is_overburden = False
fmin, fmax = 0.01, 50.0
SEM = sem2dpack(direct1)

### Read SEM2DPACK Fault Data

In [ ]:
read_fault_testing(SEM, ftag=5)
BC_bottom = SEM.fault

read_fault_testing(SEM, ftag=7)
BC_right = SEM.fault

read_fault_testing(SEM, ftag=9)
BC_left = SEM.fault

vector_orientations = {}
vector_orientations['Left'] = -1, -1
vector_orientations['Bottom'] = -1, +1
vector_orientations['Right'] = +1, +1

X_coord_L = BC_left['x']
X_coord_B = BC_bottom['x']
X_coord_R = BC_right['x']

Z_coord_L = BC_left['z']
Z_coord_B = BC_bottom['z']
Z_coord_R = BC_right['z']

Slip_1_L = BC_left['Slip_1']
Slip_1_B = BC_bottom['Slip_1']
Slip_1_R = BC_right['Slip_1']

Slip_2_L = BC_left['Slip_2']
Slip_2_B = BC_bottom['Slip_2']
Slip_2_R = BC_right['Slip_2']

time_fault = BC_left['Time']

fault_components = {}
fault_components["Left"] = X_coord_L, Z_coord_L, Slip_1_L, Slip_2_L, 'z'
fault_components["Bottom"] = X_coord_B, Z_coord_B, Slip_1_B, Slip_2_B, 'x'
fault_components["Right"] = X_coord_R, Z_coord_R, Slip_1_R, Slip_2_R, 'z'


    

### Read SEM2DPACK Station Data

In [ ]:
SEM.read_seismo('x')
time_stations = SEM.time
Vx = SEM.velocity[:,:]
Ux = np.zeros(Vx.shape)
for i in range(Vx.shape[1]):
    V = Vx[:,i]
    Ux[:,i] = compute_displacement(V, time_stations)

SEM.read_seismo('z')
Vz = SEM.velocity[:,:]
Uz = np.zeros(Vz.shape)
for i in range(Vz.shape[1]):
    V = Vz[:,i]
    Uz[:,i] = compute_displacement(V, time_stations)

XSTA, ZSTA = SEM.rcoord[:,0], SEM.rcoord[:,1]

components = {}
components['x'] = time_stations, Ux, Vx, 'blue'
components['z'] = time_stations, Uz, Vz, 'red'


### Import and Read FLAC Data

In [ ]:
def filter(signal, fc, fe, order=4):
    """
    Applies a low-pass Butterworth filter to the signal.
    """
    b, a = butter(order, fc / (0.5 * fe), btype='low')
    return filtfilt(b, a, signal)    

def read_FLAC_signal(file_path, fmax):
    """
    Reads a signal from a TXT file and returns the time and filtered signal.
    """
    data = pd.read_table(file_path, skiprows=2, sep='\s+')
    time = data.iloc[:, 0].to_numpy()
    signal = data.iloc[:, 1].to_numpy()
    dt = time[1] - time[0]
    signal = filter(signal, fmax, fe=1/dt)
    return time, signal


def read_FLAC_fault_signal(file_path, fmax):
    """
    Reads a fault signal from a TXT file and returns the time, normal displacement, and shear displacement.
    """
    data = pd.read_table(file_path)
    time = data.iloc[:, 0].to_numpy()
    normal_disp = data.iloc[:, 1].to_numpy()
    shear_disp = data.iloc[:, 2].to_numpy()
    dt = time[1] - time[0]
    normal = filter(normal_disp, fmax, fe=1/dt)
    shear = filter(shear_disp, fmax, fe=1/dt)
    return time, normal, shear


# Read FLAC fault signals
time_fault_FLAC, Ux_L, Uz_L = read_FLAC_fault_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Left_Interface.txt", 4)
_, Uz_B, Ux_B = read_FLAC_fault_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Bottom_Interface.txt", 4)
_, Ux_R, Uz_R = read_FLAC_fault_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Right_Interface.txt", 4)

# Read FLAC station signals
time_stations_FLAC, Ux_L_S = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Ux_Left.txt", 4)
_, Uz_L_S = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Uz_Left.txt", 4)
_, Ux_B_S = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Ux_Bottom.txt", 4)
_, Uz_B_S = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Uz_Bottom.txt", 4)
_, Ux_R_S = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Ux_Right.txt", 4)
_, Uz_R_S = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Uz_Right.txt", 4)

# Read FLAC input signal
time_input_FLAC, V_input_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Input.txt", 4)



# 3) Preview

### Plot Input Signal

In [ ]:
#Get SEM2DPACK Input signal path
input_signal_path = SEM.directory + '/' + 'SourcesTime_sem2d.tab'

#Plot SEM2DPACK input signal VS FLAC input signal
U_input_FLAC = compute_displacement(V_input_FLAC,time_input_FLAC)
ax_V, ax_D = plot_input_signal(input_signal_path, 'VU')
ax_V.plot(time_input_FLAC, V_input_FLAC, 'r--', label='Input Velocity FLAC')
ax_D.plot(time_input_FLAC, U_input_FLAC, 'r--', label='Input Displacement FLAC')
ax_V.legend()
ax_D.legend()
plt.show()

#Write SEM2DPACK input signal to FLAC format
    #write_FLAC_input_signal_table(input_signal_path)


### Drawing of the stations

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(4, 4))

#Desired stations
stations_left = [get_nearest_station(-12,0, XSTA, ZSTA),get_nearest_station(-12,-15, XSTA, ZSTA),get_nearest_station(-12,-30, XSTA, ZSTA)]
stations_bottom = [get_nearest_station(-10,-32, XSTA, ZSTA),get_nearest_station(0,-32, XSTA, ZSTA),get_nearest_station(10,-32, XSTA, ZSTA)]
stations_right = [get_nearest_station(12,0, XSTA, ZSTA),get_nearest_station(12,-15, XSTA, ZSTA),get_nearest_station(12,-30, XSTA, ZSTA)]
stations = np.concatenate([stations_left, stations_bottom, stations_right])

stations_dic = {}
stations_dic["Left"] = stations_left
stations_dic["Bottom"] = stations_bottom
stations_dic["Right"] = stations_right

#Desired fault locations
stations_left_fault = [12, 7, 0]
stations_bottom_fault = [0, 4, 8]
stations_right_fault = [12, 7, 0]

stations_fault_dic = {}
stations_fault_dic["Left"] = stations_left_fault
stations_fault_dic["Bottom"] = stations_bottom_fault
stations_fault_dic["Right"] = stations_right_fault

#Draw the plot
draw_example(ax)
ax.scatter(XSTA, ZSTA, marker='v', color='b', label='Stations')
ax.scatter(XSTA[stations], ZSTA[stations], marker='v', color='r', label='Selected Stations')
ax.legend()
ax.set_title("Drawing of the stations")
ax.set_ylim(-80,35)
ax.set_xlim(-20,20)


# 4) Post-Processing

### Comparison between station data near interfaces (input sollicitation)

In [ ]:

def plot_comparison_stations_interfaces(comp, interf):
    """
    Plot the comparison of SEM2DPACK and FLAC displacements near a specified interface and component.
    """
    comp = comp.capitalize()
    plt.xlabel("Time (s)")
    plt.ylabel("Displacement (m)")
    plt.grid(True)
    plt.title(f"{comp} displacement near {interf} interface")
    if comp == 'X':
        if interf == 'Left':
            index = get_nearest_station(-12,0, XSTA, ZSTA)
            plt.plot(time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_2_L[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Ux_L_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Ux_L, label="FLAC interface", c='orange')
        elif interf == 'Bottom':
            index = get_nearest_station(-0,-32, XSTA, ZSTA)
            plt.plot(time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_1_B[1,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Ux_B_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Ux_B, label="FLAC interface", c='orange')
        elif interf == 'Right':
            index = get_nearest_station(12,0, XSTA, ZSTA)
            plt.plot(time_stations, Ux[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_2_R[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Ux_R_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Ux_R, label="FLAC interface", c='orange')
    elif comp == 'Z':
        if interf == 'Left':
            index = get_nearest_station(-12,0, XSTA, ZSTA)
            plt.plot(time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, -Slip_1_L[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Uz_L_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Uz_L, label="FLAC interface", c='orange')
        elif interf == 'Bottom':
            index = get_nearest_station(0,-32, XSTA, ZSTA)
            plt.plot(time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_2_B[1,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Uz_B_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Uz_B, label="FLAC interface", c='orange')
        elif interf == 'Right':
            index = get_nearest_station(12,0, XSTA, ZSTA)
            plt.plot(time_stations, Uz[:,index], label="SEM2DPACK Station", c='blue')
            # plt.plot(time_fault, Slip_1_R[7,:], label="SEM2DPACK Interface", c='green')
            plt.plot(time_stations_FLAC, Uz_R_S, label="FLAC Station", c='purple')  
            # plt.plot(time_fault_FLAC, Uz_R, label="FLAC interface", c='orange')
    plt.legend()
    plt.show()

plot_comparison_stations_interfaces('x', 'Left')
plot_comparison_stations_interfaces('x', 'Bottom')
plot_comparison_stations_interfaces('x', 'Right')
plot_comparison_stations_interfaces('z', 'Left')
plot_comparison_stations_interfaces('z', 'Bottom')
plot_comparison_stations_interfaces('z', 'Right')

     





## Signal comparison of multiple stations 

In [ ]:
# Read FLAC station signals
_, Ux_40_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_40_X.txt", 4)
_, Uz_40_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_40_Z.txt", 4)
_, Ux_50_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_50_X.txt", 4)
_, Uz_50_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_50_Z.txt", 4)
_, Ux_60_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_60_X.txt", 4)
_, Uz_60_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_60_Z.txt", 4)
_, Ux_70_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_70_X.txt", 4)
_, Uz_70_FLAC = read_FLAC_signal("C:/Users/t.guyonneau/OneDrive - EGIS Group/Documents/GitHub/sem2dpack/JUPYTER/Stage_Theo/Signals/Station_70_Z.txt", 4)

#Read SEM2DPACK station signals
Ux_40_SEM = Ux[:,get_nearest_station(0,-40, XSTA, ZSTA)]
Uz_40_SEM = Uz[:,get_nearest_station(0,-40, XSTA, ZSTA)]
Ux_50_SEM = Ux[:,get_nearest_station(0,-50, XSTA, ZSTA)]
Uz_50_SEM = Uz[:,get_nearest_station(0,-50, XSTA, ZSTA)]
Ux_60_SEM = Ux[:,get_nearest_station(0,-60, XSTA, ZSTA)]
Uz_60_SEM = Uz[:,get_nearest_station(0,-60, XSTA, ZSTA)]
Ux_70_SEM = Ux[:,get_nearest_station(0,-70, XSTA, ZSTA)]
Uz_70_SEM = Uz[:,get_nearest_station(0,-70, XSTA, ZSTA)]


def plot_stations_comparison(comp, savefig=None):
    """
    Plots the comparison of displacements at stations for a given component (x or z).
    """
    comp = comp.capitalize()
    #Creation of the figure and axes
    fig = plt.figure()
    ax_FLAC = fig.add_subplot(2, 1, 1)
    ax_SEM = fig.add_subplot(2, 1, 2)
    if comp == 'X':
        #FLAC stations
        ax_FLAC.plot(time_stations_FLAC, Ux_40_FLAC, label="FLAC Station at Z=-40m")
        ax_FLAC.plot(time_stations_FLAC, Ux_50_FLAC, label="FLAC Station at Z=-50m")
        ax_FLAC.plot(time_stations_FLAC, Ux_60_FLAC, label="FLAC Station at Z=-60m")
        ax_FLAC.plot(time_stations_FLAC, Ux_70_FLAC, label="FLAC Station at Z=-70m")
        #SEM2DPACK stations
        ax_SEM.plot(time_stations, Ux_40_SEM, label="SEM2DPACK Station at Z=-40m")
        ax_SEM.plot(time_stations, Ux_50_SEM, label="SEM2DPACK Station at Z=-50m")
        ax_SEM.plot(time_stations, Ux_60_SEM, label="SEM2DPACK Station at Z=-60m")
        ax_SEM.plot(time_stations, Ux_70_SEM, label="SEM2DPACK Station at Z=-70m")
    elif comp == 'Z':
        #FLAC stations
        ax_FLAC.plot(time_stations_FLAC, Uz_40_FLAC, label="FLAC Station at Z=-40m")
        ax_FLAC.plot(time_stations_FLAC, Uz_50_FLAC, label="FLAC Station at Z=-50m")
        ax_FLAC.plot(time_stations_FLAC, Uz_60_FLAC, label="FLAC Station at Z=-60m")
        ax_FLAC.plot(time_stations_FLAC, Uz_70_FLAC, label="FLAC Station at Z=-70m")
        #SEM2DPACK stations
        ax_SEM.plot(time_stations, Uz_40_SEM, label="SEM2DPACK Station at Z=-40m")
        ax_SEM.plot(time_stations, Uz_50_SEM, label="SEM2DPACK Station at Z=-50m")
        ax_SEM.plot(time_stations, Uz_60_SEM, label="SEM2DPACK Station at Z=-60m")
        ax_SEM.plot(time_stations, Uz_70_SEM, label="SEM2DPACK Station at Z=-70m")
        
    #Setting titles and labels
    fig.suptitle(f"{comp} Displacements of stations at X=0m")
    fig.supxlabel("Time (s)")
    fig.supylabel("Displacement (m)")
    ax_FLAC.grid(True)
    ax_SEM.grid(True)
    ax_FLAC.legend()
    ax_SEM.legend()
    plt.show()

    if savefig:
        path = 'Stations_Comparison_' + comp + '.png'
        fig.savefig(path, bbox_inches='tight', dpi=300)
        print(f"Figure saved as {path}")

plot_stations_comparison('x')
plot_stations_comparison('z')

## Comparison between fault data

In [ ]:

def plot_comparison(comp, interf, savefig=None):
    fig, ax = plt.subplots()
    comp = comp.capitalize()
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Displacement (m)")
    ax.set_title(f"{comp} displacement at {interf} Interface (Kn = {Kn:.2e} Pa/m | Ks = {Ks:.2e} Pa/m | Phi = {Phi:.2f}°)")
    ax.grid(True)
    if comp == 'X':
        if interf == 'Left':
            ax.plot(time_fault, Slip_2_L[7,:], label="SEM2DPACK Interface", c='green')
            ax.plot(time_fault_FLAC, Ux_L, label="FLAC interface", c='orange')
            ratio = np.mean(Slip_2_L[7,-10:]/Ux_L[-10:])
        elif interf == 'Bottom':
            ax.plot(time_fault, Slip_1_B[1,:], label="SEM2DPACK Interface", c='green') 
            ax.plot(time_fault_FLAC, Ux_B, label="FLAC interface", c='orange')
            ratio = np.mean(Slip_1_B[1,-10:]/Ux_B[-10:])
        elif interf == 'Right':
            ax.plot(time_fault, Slip_2_R[7,:], label="SEM2DPACK Interface", c='green') 
            ax.plot(time_fault_FLAC, Ux_R, label="FLAC interface", c='orange')
            ratio = np.mean(Slip_2_R[7,-10:]/Ux_R[-10:])
    elif comp == 'Z':
        if interf == 'Left':
            ax.plot(time_fault, -Slip_1_L[7,:], label="- SEM2DPACK Interface", c='green')
            ax.plot(time_fault_FLAC, Uz_L, label="FLAC interface", c='orange')
            ratio = np.mean(-Slip_1_L[7,-10:]/Uz_L[-10:])
        elif interf == 'Bottom':
            ax.plot(time_fault, Slip_2_B[1,:], label="SEM2DPACK Interface", c='green')
            ax.plot(time_fault_FLAC, Uz_B, label="FLAC interface", c='orange')
            ratio = np.mean(Slip_2_B[1,-10:]/Uz_B[-10:])
        elif interf == 'Right':
            ax.plot(time_fault, Slip_1_R[7,:], label="SEM2DPACK Interface", c='green')
            ax.plot(time_fault_FLAC, Uz_R, label="FLAC interface", c='orange')
            ratio = np.mean(Slip_1_R[7,-10:]/Uz_R[-10:])
    ax.legend()
    plt.show()

    print(f"SEM/FLAC Ratio : {ratio:.2f}")

    if savefig:
        path = 'U_' + comp + '_' + interf + '.png'
        fig.savefig(path, bbox_inches='tight', dpi=300)
        print(f"Figure saved as {path}")

plot_comparison('x', 'Left', savefig=True)
plot_comparison('x', 'Bottom', savefig=True)
plot_comparison('x', 'Right', savefig=True)
plot_comparison('z', 'Left', savefig=True)
plot_comparison('z', 'Bottom', savefig=True)
plot_comparison('z', 'Right', savefig=True)
     



